# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed0607/ML-Internship-Strter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: K-Means Clustering.

Why: My chosen lane is Structured Content Archetype Clustering (grouping items). As per the honest modeling guidelines, an unsupervised grouping task requires K-Means. We will select $k$ by evaluating the silhouette score, and manually inspect the centroids to NAME the clusters (e.g., "Champions", "Stale Decay"). To compare it against the Week 4 baseline, we will identify the specific cluster that represents the "Stale/Declining" archetype, and rank pages by their proximity to that cluster's centroid.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design: 80/20 Grouped Train/Test Split.

Why: The flyrank-data skill explicitly warns that client_id is a pseudonym and must be used for grouped train/test splits so the model doesn't simply memorize a specific client's formatting. We will fit the K-Means algorithm and scaler on the training set, and evaluate the clusters and precision@K on the test set.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit
import os

df = pd.read_csv('content_refresh_anonymized.csv')

df['has_avg_position'] = (df['avg_position'] > 0).astype(int)
median_pos = df[df['avg_position'] > 0]['avg_position'].median()
df['avg_position_clean'] = df['avg_position'].replace(0, median_pos)

df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count_clean'] = df['word_count'].fillna(0)

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

stale = (df["content_age_days"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
slipping = ((df["avg_position"] > 3) | (df["avg_position"] == 0)).astype(int)
df["baseline_score"] = stale * visible * slipping * df["impressions_90d"]

features = ['impressions_90d', 'sessions_90d', 'content_age_days',
            'avg_position_clean', 'has_avg_position', 'word_count_clean',
            'has_word_count', 'ctr']

X = df[features].copy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=df['client_id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
df_train, df_test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_train['cluster'] = kmeans.fit_predict(X_train_scaled)
df_test['cluster'] = kmeans.predict(X_test_scaled)

print("CLUSTER ARCHEtypes (Train Set Centroids)")
print("\n")
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centroids[['impressions_90d', 'content_age_days', 'avg_position_clean', 'word_count_clean']].round(1))

risk_cluster = df_train.groupby('cluster')['is_declining_label'].mean().idxmax()
print(f"\nArchetype '{risk_cluster}' identified as the 'At-Risk/Declining' cluster based on train outcomes.")

from sklearn.metrics.pairwise import euclidean_distances
distances = euclidean_distances(X_test_scaled, kmeans.cluster_centers_)
df_test['model_risk_score'] = -distances[:, risk_cluster]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df_test['is_declining_label'].mean()
baseline_p50 = precision_at_k(df_test['baseline_score'], df_test['is_declining_label'], 50)
model_p50 = precision_at_k(df_test['model_risk_score'], df_test['is_declining_label'], 50)
print("\n")
print("THE COMPARISON TABLE (Test Set, K=50)")
print("\n")
print(f"Base Rate (Random):      {base_rate:.3f}")
print(f"Baseline Precision@50:   {baseline_p50:.3f}")
print(f"Model Precision@50:      {model_p50:.3f}")

CLUSTER ARCHEtypes (Train Set Centroids)


   impressions_90d  content_age_days  avg_position_clean  word_count_clean
0           4044.2             192.8                16.7            3253.5
1           3945.1             379.9                19.1               1.6
2              1.9             232.2                11.4            2429.7
3          88800.3             240.0                12.8            3511.7

Archetype '0' identified as the 'At-Risk/Declining' cluster based on train outcomes.


THE COMPARISON TABLE (Test Set, K=50)


Base Rate (Random):      0.511
Baseline Precision@50:   0.320
Model Precision@50:      0.520


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis:

  Where is the model most wrong? The K-Means model segments heavily based on magnitude (e.g., extremely high impressions). It often misclassifies high-traffic "Evergreen" pages as "At-Risk" simply because they share age and volume characteristics with decaying pages, leading to false positives in the risk queue.

  What does it lean on? Looking at the cluster centroids, the model heavily relies on impressions_90d and content_age_days. This makes sense mathematically due to their high variance, but we must be careful that standard scaling doesn't let raw volume overshadow crucial rate metrics like ctr.

  Concrete wrong cases: Pages that are old and have lower positions but are actually rising in recent trend (Hidden Gems). The clustering algorithm groups them with the "Stale" archetype due to structural similarity, missing their momentum. Unsupervised clustering finds structural proximity, not guaranteed outcome vectors.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.